# Exercise: Oxygen binding to Cu(111) surfaces

In this exercise we are going to leverage some pre-built tools in the Atomic Simulation Environment (ASE) package. In particular, we are going to use ASE's build module to construct a Cu(111) surface slab, and then we are going to perform structural optimizations for various oxygen adsorption sites using the MACE-MP Machine-Learned Interatomic Potential. We will evaluate the binding energy at these adsorption sites and compare them to an experimental value.

One advantage in constructing our surface using ase.build.fcc111 is that adsorption sites are already pre-built into the object. It is then trivial to add a single adsorbate oxygen atom with ase.build.add_adsorbate. The alternative is to construct the ase.Atoms object manually, and then manually placing adsorbate oxygen atoms.

The goal of this exercise is to gain experience using the ASE package with Machine-Learned Interatomic Potentials. AIMNet2, MACE, and UMA have interfaces to ASE.

We begin by importing everything we need and building our Cu(111) slab

In [1]:
from ase import Atoms
from ase.build import fcc111, add_adsorbate
from ase.optimize import BFGS
from ase.visualize import view

from mace.calculators import MACECalculator


# Build and view our Cu(111) slab
slab = fcc111('Cu', size=(2,2,3))
view(slab, viewer='x3d')

/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


## For each adsorption site (ontop, bridge, fcc, hcp)

(1) Copy our original slab object 

(2) Add an oxygen atom to that adsorption site

(3) Since our system has periodic boundaries, we will add a vacuum above the surface of the slab

In [2]:
ontop = slab.copy()
add_adsorbate(ontop, 
              adsorbate='O', 
              height=1.5, 
              position='ontop')
ontop.center(vacuum=10.0, axis=2)
view(ontop, viewer='x3d')

In [3]:
bridge = slab.copy()
add_adsorbate(bridge, 
              adsorbate='O', 
              height=1.0, 
              position='bridge')
bridge.center(vacuum=10.0, axis=2)
view(bridge, viewer='x3d')

In [4]:
hcp = slab.copy()
add_adsorbate(hcp, 
              adsorbate='O', 
              height=1.5, 
              position='hcp')
hcp.center(vacuum=10.0, axis=2)
view(hcp, viewer='x3d')

In [5]:
fcc = slab.copy()
add_adsorbate(fcc,
              adsorbate='O', 
              height=1.5, 
              position='fcc')
fcc.center(vacuum=10.0, axis=2)
view(fcc, viewer='x3d')

## Optimize each structure

We must first setup a calculator object and then attach it to each of our systems

In [6]:
# https://mace-docs.readthedocs.io/en/latest/guide/foundation_models.html
#  MODEL is MACE-MP-0 medium
MODEL = './2023-12-03-mace-128-L1_epoch-199.model'
calculator = MACECalculator(model_paths=MODEL, 
                            device='cpu',   # 'cuda' for GPU
                            default_dtype='float64')

ontop.calc = calculator
bridge.calc = calculator
hcp.calc = calculator
fcc.calc = calculator

/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


Optimize each system to 0.001 eV/angstrom

In [7]:
for system in [ontop, bridge, hcp, fcc]:
    opt = BFGS(system)
    opt.run(fmax=0.001)

      Step     Time          Energy          fmax
BFGS:    0 16:35:15      -48.838193       11.163249
BFGS:    1 16:35:16      -49.929711        0.946152
BFGS:    2 16:35:16      -49.947686        0.666574
BFGS:    3 16:35:16      -49.962718        0.371564
BFGS:    4 16:35:16      -49.965140        0.154839
BFGS:    5 16:35:17      -49.967059        0.087951
BFGS:    6 16:35:17      -49.971229        0.138608
BFGS:    7 16:35:17      -49.972383        0.089948
BFGS:    8 16:35:17      -49.973040        0.063599
BFGS:    9 16:35:17      -49.973448        0.045737
BFGS:   10 16:35:18      -49.973783        0.051638
BFGS:   11 16:35:18      -49.973905        0.027996
BFGS:   12 16:35:18      -49.973926        0.009013
BFGS:   13 16:35:18      -49.973933        0.007967
BFGS:   14 16:35:18      -49.973942        0.013083
BFGS:   15 16:35:18      -49.973952        0.012808
BFGS:   16 16:35:19      -49.973958        0.007856
BFGS:   17 16:35:19      -49.973962        0.005443
BFGS:   18 16:

Unfortunately, the bridge site relaxes into the fcc position.

## Calculate oxygen binding energies

We can calculate oxygen binding energy to each site
$$
E_{\mathrm{bind}}
=
E_{S+A}
-
E_S
-
E_A
$$

$ E_{S+A} $ is the energy of the surface+adsorbate system [Cu(111)+O]

$ E_S $ is the energy of the surface [clean Cu(111)]

$ E_A $ is the energy of the gas-phase adsorbate [in our case, atomic oxygen]


In [8]:
# Get gas-phase atomic oxygen energy
O = Atoms('O', positions=[[0, 0, 0]])
O.calc = calculator
O_energy = O.get_potential_energy()


In [9]:
# Optimize our slab
slab = fcc111('Cu', size=(2,2,3))
slab.center(vacuum=10.0, axis=2)
slab.calc = calculator
opt = BFGS(slab)
opt.run(fmax=0.001)
view(slab,viewer='x3d')

      Step     Time          Energy          fmax
BFGS:    0 16:39:22      -45.351450        0.034923
BFGS:    1 16:39:22      -45.351585        0.032801
BFGS:    2 16:39:22      -45.352608        0.000505


# Compute binding energies

In [10]:
def compute_binding_energy(adsorbed_system, clean_surface, adsorbate):
    '''
    Compute the binding energy of an adsorbate on a surface.

    Parameters:
    adsorbed_system : ASE Atoms object
        The system with the adsorbate on the surface.
    clean_surface : ASE Atoms object
        The clean surface without the adsorbate.
    adsorbate : ASE Atoms object
        The isolated adsorbate.

    Returns:
    float
        The binding energy of the adsorbate on the surface.
    '''
    E_SA = adsorbed_system.get_potential_energy()
    E_S = clean_surface.get_potential_energy()
    E_O = adsorbate.get_potential_energy()
    return E_SA - E_S - E_O

In [11]:
E_binding_ontop = compute_binding_energy(ontop, slab, O)
#E_binding_bridge = compute_binding_energy(bridge, slab, O)
E_binding_fcc = compute_binding_energy(fcc, slab, O)
E_binding_hcp = compute_binding_energy(hcp, slab, O)

print(f"Binding energy (ontop): {E_binding_ontop:>.3f} eV")
print(f"Binding energy (fcc): {E_binding_fcc:>.3f} eV")
print(f"Binding energy (hcp): {E_binding_hcp:>.3f} eV")
print()
print('Experimental value')
print('E. Shustorovich and A. T. Bell, Surf. Sci. 268, 397 (1992).https://doi.org/10.1016/0039-6028(92)90979-G')
print('Binding energy: -4.47 eV')

Binding energy (ontop): -2.540 eV
Binding energy (fcc): -4.397 eV
Binding energy (hcp): -4.380 eV

Experimental value
E. Shustorovich and A. T. Bell, Surf. Sci. 268, 397 (1992).https://doi.org/10.1016/0039-6028(92)90979-G
Binding energy: -4.47 eV
